# H4: The Weekend Effect — Do Fridays Outperform Mondays?
## S&P 500 — Quantitative Research Series

**Author:** Furkan Çelik  
**Data:** Custom PostgreSQL pipeline · yfinance API · ~500 equities · Full history  
**Tools:** Python, PostgreSQL, SQLAlchemy, SciPy, Seaborn

---

### 1. Hypothesis Definition

| | |
|---|---|
| **Research Question** | Is there a statistically significant difference between Friday and Monday daily log returns across the entire S&P 500 universe? |
| **H₀ (Null)** | The mean daily log return on Fridays equals that on Mondays. |
| **H₁ (Alternative)** | Friday daily log returns are significantly **greater** than Monday log returns. |

### 2. Data & Methodology

| | |
|---|---|
| **Population** | S&P 500 constituents · Full available history |
| **Variables** | Daily logarithmic returns: ln(Pₜ / Pₜ₋₁) × 100, computed via SQL LAG() window function |
| **Statistical Test** | Welch's Independent Samples T-Test (one-tailed, alternative='greater') + Distribution comparison (KDE) |
| **Additional Metrics** | Skewness, Excess Kurtosis (Fisher) for both groups |
| **Significance Level** | α = 0.05 |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from db_utils import fetch_data

sns.set_theme(style="darkgrid")
plt.rcParams.update({'figure.figsize': (13, 6), 'figure.dpi': 120})
print("✅ Environment ready.")


### 3. Data Extraction

In [ ]:
# SQL: Compute daily log returns and filter only Fridays (5) & Mondays (1)
# LAG() computes the previous closing price within each ticker's time series
query = """
WITH DailyReturns AS (
    SELECT
        ticker,
        date,
        EXTRACT(ISODOW FROM date) AS day_of_week,
        LN(close / LAG(close) OVER (PARTITION BY ticker ORDER BY date)) * 100 AS log_return
    FROM daily_prices
)
SELECT day_of_week, log_return
FROM DailyReturns
WHERE log_return IS NOT NULL
  AND day_of_week IN (1, 5)
"""

print("Fetching Friday and Monday log returns from full history (may take a moment)...")
df = fetch_data(query)
print(f"Dataset: {len(df):,} daily observations loaded  (Mon: {(df['day_of_week']==1).sum():,} | Fri: {(df['day_of_week']==5).sum():,})")


### 4. Statistical Analysis & Results

In [ ]:
friday  = df[df['day_of_week'] == 5]['log_return'].dropna()
monday  = df[df['day_of_week'] == 1]['log_return'].dropna()

# T-Test
t_stat, p_value = stats.ttest_ind(friday, monday, alternative='greater', equal_var=False)

# Distribution stats
mu_fri,  mu_mon  = friday.mean(),  monday.mean()
sk_fri,  sk_mon  = stats.skew(friday),  stats.skew(monday)
kurt_fri, kurt_mon = stats.kurtosis(friday, fisher=True), stats.kurtosis(monday, fisher=True)

h4_result = "✅ CONFIRMED" if p_value < 0.05 else "❌ REJECTED"
diff = mu_fri - mu_mon
tradeable = "Likely No" if diff < 0.05 else "Potentially — run cost analysis"

verdict = f"""
### 5. Statistical Findings

| Metric | Friday | Monday |
|---|---|---|
| **N (observations)** | {len(friday):,} | {len(monday):,} |
| **Mean Log Return** | **%{mu_fri:.4f}** | **%{mu_mon:.4f}** |
| **Skewness** | {sk_fri:.3f} | {sk_mon:.3f} |
| **Excess Kurtosis** | {kurt_fri:.3f} | {kurt_mon:.3f} |

| Test Metric | Value |
|---|---|
| **T-Statistic** | {t_stat:.4f} |
| **p-value (one-tailed)** | {p_value:.4e} |
| **H₀ Decision** | {'Rejected (p < 0.05)' if p_value < 0.05 else 'Not Rejected (p ≥ 0.05)'} |

### 6. Quant Verdict

| | |
|---|---|
| **Result** | **{h4_result}** |
| **Mean Return Difference** | %{diff:.4f} (Friday minus Monday) |
| **Interpretation** | {'A statistically significant Friday-Monday return gap exists across the S&P 500 universe.' if p_value < 0.05 else 'No statistically significant difference between Friday and Monday returns. The Weekend Effect is **not** present in this dataset.'} |
| **Tradeable after costs?** | **{tradeable}** — A %{diff:.4f} daily edge is unlikely to survive round-trip transaction costs + slippage at scale. |
| **ML Feature Value** | {'Day-of-week should be included as a categorical feature in intraday/swing models.' if p_value < 0.05 else 'Day-of-week does not add predictive value based on this test.'} |
"""

display(Markdown(verdict))

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# KDE plot (zoomed in)
ax = axes[0]
fri_z = friday[(friday > -3) & (friday < 3)]
mon_z = monday[(monday > -3) & (monday < 3)]
sns.kdeplot(mon_z,  fill=True, color="#E17055", label=f"Monday  (μ=%{mu_mon:.3f})",  ax=ax)
sns.kdeplot(fri_z, fill=True, color="#00B894", label=f"Friday (μ=%{mu_fri:.3f})", ax=ax)
ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.set_title("Daily Log Return Distribution (KDE)
±3% zoomed", fontsize=12)
ax.set_xlabel("Daily Log Return (%)")
ax.set_ylabel("Density")
ax.legend()

# Box plot
ax = axes[1]
sns.boxplot(data=[monday, friday], palette=["#E17055", "#00B894"], showfliers=False, ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels([f"Monday\n(μ=%{mu_mon:.3f})", f"Friday\n(μ=%{mu_fri:.3f})"], fontsize=12)
ax.axhline(0, color='gray', linestyle='--')
ax.set_ylabel("Daily Log Return (%)")
ax.set_title("Return Distribution — Box Plot
(Outliers hidden)", fontsize=12)

plt.suptitle(f"H4: Weekend Effect — Welch's T-Test | p = {p_value:.4e}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
